In [ ]:
# Install hypertools (dev-1.0 preview) -- run this first on Colab.
# On release this becomes: %pip install hypertools
%pip install -q "hypertools[interactive] @ git+https://github.com/ContextLab/hypertools.git@dev-1.0"

# Dimensionality reduction

The `reduce` function reduces the dimensionality of an array or list of arrays. The default model is IncrementalPCA, but a variety of models are supported. Note that `ndims` defaults to `None`, which means no dimensionality reduction is performed unless you explicitly request a target number of dimensions via `ndims`.

Supported models include: PCA, IncrementalPCA, SparsePCA, MiniBatchSparsePCA, KernelPCA, FastICA, FactorAnalysis, TruncatedSVD, DictionaryLearning, MiniBatchDictionaryLearning, TSNE, Isomap, SpectralEmbedding, LocallyLinearEmbedding, MDS and UMAP.

## Import Hypertools

In [2]:
import hypertools as hyp

%matplotlib inline

## Load your data

First, we'll load one of the sample datasets. This dataset is a list of 2 `numpy` arrays, each containing average brain activity (fMRI) from 18 subjects listening to the same story, fit using Hierarchical Topographic Factor Analysis (HTFA) with 100 nodes.  The rows are timepoints and the columns are fMRI components. 

See the [full dataset](http://dataspace.princeton.edu/jspui/handle/88435/dsp015d86p269k) or the [HTFA article](https://www.biorxiv.org/content/early/2017/02/07/106690) for more info on the data and HTFA, respectively. 

In [3]:
weights = hyp.load('weights_avg')

## Reduce one array

Let's look at one array from the dataset above.

In [4]:
print('Array shape: (%d, %d)' % weights[0].shape)

Array shape: (100, 100)


To reduce this array, pass the array to `hyp.reduce` along with the desired number of dimensions via `ndims`. Below we reduce from 100 features to 3 features by setting `ndims=3` (if `ndims` is omitted, no reduction is performed).

In [5]:
reduced_array = hyp.reduce(weights[0], ndims=3)
print('Reduced array shape: (%d, %d)' % reduced_array.shape)

Reduced array shape: (100, 3)


## Reduce list of arrays

A list or numpy array of multiple arrays can also be reduced into a common space. That is, the data can be combined, reduced as a whole, then split back into individual elements and outputted via hyp.reduce.  

Here we show this with the two arrays in the weights dataset. The first was printed above as a 100 x 100 array (timepoints by components); the second has the same shape.

Now, let's reduce both arrays at once (by passing in the whole of the weights data) and re-examine the data.

In [6]:
reduced_arrays = hyp.reduce(weights, ndims=3)
print('Shape of first reduced array: ', reduced_arrays[0].shape)
print('Shape of second reduced array: ', reduced_arrays[1].shape)

Shape of first reduced array:  (100, 3)
Shape of second reduced array:  (100, 3)


We can see that each array has been reduced from 100 features to 3 features (since we set `ndims=3`), with the number of datapoints unchanged.

## Reduce list of arrays (TSNE)

You can also opt to use different reduction methods. In the example below, we reduce multiple arrays at once, using TSNE, again passing `ndims=3` to reduce to three dimensions.

TSNE is a nonlinear neighbor embedding: it preserves each point's local neighborhood rather than the directions of greatest variance, so distances between far-apart points in the embedding are not meaningful. It also provides no reusable projection (scikit-learn's `TSNE` has no `transform` method), so a new dataset cannot be mapped into an existing embedding, and the result depends on the initialization: with scikit-learn's current default (`init='pca'`) repeated runs match, but with `init='random'` each run gives a different embedding unless you pass `random_state`.

In [7]:
reduced_TSNE = hyp.reduce(weights, reduce='TSNE', ndims=3)
print('Shape of first reduced array: ',reduced_TSNE[0].shape)
print('Shape of second reduced array: ',reduced_TSNE[1].shape)

Shape of first reduced array:  (100, 3)
Shape of second reduced array:  (100, 3)


## Reduce to specified number of dimensions

You may prefer to reduce to a specific number of features other than three. To achieve this, simply pass the number of desired features (as an int) to the ndims argument, as below.

In [8]:
reduced_4 = hyp.reduce(weights, ndims = 4)
print('Shape of first reduced array: ', reduced_4[0].shape)
print('Shape of second reduced array: ', reduced_4[1].shape)

Shape of first reduced array:  (100, 4)
Shape of second reduced array:  (100, 4)


## Reduce list of arrays with specific parameters

For finer control of parameters, pass the reduce argument a dictionary with the keys `model` (the desired reduction method) and `kwargs` (a dictionary of model parameters). See [scikit-learn](http://scikit-learn.org/stable/index.html) model docs for details on parameters supported for each model.

Supported models include: PCA, IncrementalPCA, SparsePCA, MiniBatchSparsePCA, KernelPCA, FastICA, FactorAnalysis, TruncatedSVD, DictionaryLearning, MiniBatchDictionaryLearning, TSNE, Isomap, SpectralEmbedding, LocallyLinearEmbedding, MDS, and UMAP.

The example below reduces to three features by also passing `ndims=3` alongside the dictionary of model parameters.

Below, `whiten=True` rescales each component to unit variance, so the returned features have equal spread rather than the first principal component dominating the range. The shape of the data is preserved; the relative scale of the components is not.

In [9]:
reduced_params = hyp.reduce(weights, reduce={'model': 'PCA', 'kwargs': {'whiten': True}}, ndims=3)